# Create a random MLP/data and create Jacobian for randomized SVD

In [148]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
from flax.linen import initializers
# Really cool package for prototyping. 
import treescope
treescope.basic_interactive_setup()

In [279]:
# Might be able to use ravel eventually but this is easier
def flatten_jacobian(jacobian_dict, x):
    batch_size = x.shape[0]
    flat_jacobian = []
    for layer in jacobian_dict['params'].values():
        for param in layer.values():
            flat_jacobian.append(param.reshape(batch_size, -1))
    return jnp.concatenate(flat_jacobian, axis=1)

# Create standard model 
class MLP(nn.Module):
    num_layers: int = 4
    hidden_dim: int = 10

    @nn.compact
    def __call__(self, x):
        # Input x is expected to be of shape (batch_size, 1)
        for _ in range(self.num_layers):
            x = nn.Dense(features=self.hidden_dim, use_bias=True,
                    kernel_init=initializers.kaiming_uniform(),  # Kaiming uniform initialization
                    bias_init=initializers.zeros  # Bias initialized to zero
            )(x)
            x = nn.tanh(x)  # Apply activation function after each layer
        x = nn.Dense(features=1)(x)
        return x
        
# Initialize the model and parameters
key = jax.random.PRNGKey(0)
input_shape = (16, 1)  # Example input shape for MNIST
x = jax.random.normal(key, input_shape)

model = MLP()
params = model.init(key, x)

# Define the function f(x, theta)
def f(params, x):
    return model.apply(params, x)

# Compute the Jacobian with respect to the weights
jacobian_fn = jax.jacrev(f, argnums=0)
jacobian = flatten_jacobian(jacobian_fn(params, x), x)
print(jacobian.shape)
print(model.tabulate(
    jax.random.key(0), x, compute_flops=True, compute_vjp_flops=True
))

(16, 361)

                                  MLP Summary                                   
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ path    ┃ module ┃ inputs     ┃ outputs    ┃ flops ┃ vjp_flops ┃ params      ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│         │ MLP    │ float32[1… │ float32[1… │ 0     │ 0         │             │
├─────────┼────────┼────────────┼────────────┼───────┼───────────┼─────────────┤
│ Dense_0 │ Dense  │ float32[1… │ float32[1… │ 0     │ 0         │ bias:       │
│         │        │            │            │       │           │ float32[10] │
│         │        │            │            │       │           │ kernel:     │
│         │        │            │            │       │           │ float32[1,… │
│         │        │            │            │       │           │             │
│         │        │            │            │       │           │ 20 (80 B)   │
├─────────┼──────

# Calculate SVD the regular way 

In [280]:
u_true, s_true, vh_true = jnp.linalg.svd(jacobian)
u_true, s_true, vh_true

(<jax.Array float32(16, 16) ≈0.021 ±0.25 [≥-0.7, ≤0.67] nonzero:256
   <Arrayviz rendering>
 | Device: GPU 0>,
 <jax.Array float32(16,) ≈1.5 ±3.1 [≥3.3e-05, ≤1.2e+01] nonzero:16
   <Arrayviz rendering>
 | Device: GPU 0>,
 <jax.Array float32(361, 361) ≈0.0025 ±0.053 [≥-0.62, ≤1.0] nonzero:130_321
   <Arrayviz rendering>
 | Device: GPU 0>)

# Calculate SVD the randomized way

https://arxiv.org/pdf/0909.4061

## First Randomized Range Finder Algo 4.1

https://galton.uchicago.edu/~lekheng/meetings/mmds/slides2010/Martinsson.pdf slide 9

In [281]:
key = jax.random.PRNGKey(0)
l = 5

A = jacobian
m, n = A.shape

# Step 1: Draw an n x l Gaussian random matrix Ω
Omega = jax.random.normal(key, (n, l))

# Step 2: Form the m x l matrix Y = AΩ
Y = jnp.dot(A, Omega)

# Step 3: Construct an m x l matrix Q whose columns form an orthonormal basis for the range of Y
print(f'{Y.shape}')
Q, _ = jnp.linalg.qr(Y)

# Step 4: Form the k x n matrix B = Q^T A
B = Q.T @ A

# Step 5
print(f'{B.shape}')
Uhat, s_est, vh_est = jnp.linalg.svd(B)

# Step 6
U_est = Q @ Uhat


(16, 5)
(5, 361)


In [282]:
U_est, s_est, vh_est

(<jax.Array float32(16, 5) ≈-0.064 ±0.24 [≥-0.71, ≤0.37] nonzero:80
   <Arrayviz rendering>
 | Device: GPU 0>,
 <jax.Array float32(5,) ≈4.5 ±4.3 [≥0.85, ≤1.2e+01] nonzero:5
   <Arrayviz rendering>
 | Device: GPU 0>,
 <jax.Array float32(361, 361) ≈0.0027 ±0.053 [≥-0.55, ≤1.0] nonzero:130_321
   <Arrayviz rendering>
 | Device: GPU 0>)

## Algorithm 4.2 

Iterative 

In [283]:
A = jacobian.T

epsilon = 1e-2
r = 10 

m, n = A.shape
keys = jax.random.split(key, r)

# Step 1: Draw standard Gaussian vectors ω(1), ..., ω(r) of length n
Omega = jnp.stack([jax.random.normal(keys[i], (n,)) for i in range(r)], axis=1)

# Step 2: For i = 1, 2, ..., r, compute y(i) = Aω(i)
Y = jnp.dot(A, Omega)
print(f'{Y.shape=}')

# Step 3: Initialize j and Q(0)
j = 0
Q = jnp.zeros((m, 0))

# Step 5: While loop condition
# print(Y[:, j+1:].shape)
while jnp.max(jnp.linalg.norm(Y[:, j+1:], axis=0)) > epsilon / (10 * jnp.sqrt(2 / jnp.pi)):
    # Step 6: Increment j
    # print(j)
    j += 1
    
    # Step 7: Overwrite y(j) by (I - Q(j-1)(Q(j-1))*)y(j)
    y_j = Y[:, j-1]
    if Q.shape[1] > 0:
        y_j = y_j - jnp.dot(Q, jnp.dot(Q.T, y_j))
    
    # Step 8: Normalize y(j) to get q(j)
    q_j = y_j / jnp.linalg.norm(y_j)
    
    # Step 9: Update Q(j)
    Q = jnp.hstack((Q, q_j[:, None]))
    
    # Step 10: Draw a new standard Gaussian vector ω(j+r) of length n
    key, subkey = jax.random.split(key)
    omega_jr = jax.random.normal(subkey, (n,))
    
    # Step 11: Compute y(j+r) = (I - Q(j)(Q(j))*) Aω(j+r)
    y_jr = jnp.dot(A, omega_jr)
    y_jr = y_jr - jnp.dot(Q, jnp.dot(Q.T, y_jr))
    
    # Step 12-14: Orthogonalize y(i) for i = (j + 1), ..., (j + r - 1)
    for i in range(j, j + r - 1):
        y_i = Y[:, i]
        y_i = y_i - q_j * jnp.dot(q_j, y_i)
        Y = Y.at[:, i].set(y_i)
    
    # Update Y with the new y(j+r)
    Y = jnp.hstack((Y, y_jr[:, None]))

Q

Y.shape=(361, 10)


<jax.Array float32(361, 16) ≈0.0016 ±0.053 [≥-0.46, ≤0.53] nonzero:5_776
  <Arrayviz rendering>
| Device: GPU 0>

In [284]:
# Now we can calculate 

# Step 4: Form the k x n matrix B = Q^T A
B = Q.T @ A

# Step 5
print(f'{B.shape}')
Uhat, s_est, vh_est = jnp.linalg.svd(B)

# Step 6
U_est = Q @ Uhat


(16, 16)


In [285]:
# jnp.abs(s_est[0:14] - s_true[0:14])/s_true[0:14]

In [286]:
s_est

<jax.Array float32(16,) ≈1.5 ±3.1 [≥2.4e-06, ≤1.2e+01] nonzero:16
  <Arrayviz rendering>
| Device: GPU 0>

## ALgorithm 4.2; iterative 

Gonna change so that we don't precalculate anything... 

In [287]:
A = jacobian

r = 10 

m, n = A.shape
keys = jax.random.split(key, r)

# Step 1: Draw standard Gaussian vectors ω(1), ..., ω(r) of length n
Omega = jnp.stack([jax.random.normal(keys[i], (n,)) for i in range(r + 1)], axis=1)

# Initialize initial block 
Q = jnp.zeros((m, 0))

# Step 5: While loop condition
for j in range(1, r+1):    
    # Generate y = A\omega
    y_j = jnp.dot(A, Omega[:, j])
    
    y_j = y_j - jnp.dot(Q, jnp.dot(Q.T, y_j))
    
    q_j = y_j / jnp.linalg.norm(y_j)
    
    Q = jnp.hstack((Q, q_j[:, None]))
    

In [288]:
# Now we can calculate 

# Step 4: Form the k x n matrix B = Q^T A
B = Q.T @ A

# Step 5
print(f'{B.shape}')
Uhat, s_est, vh_est = jnp.linalg.svd(B)

# Step 6
U_est = Q @ Uhat


(10, 361)


In [289]:
jnp.abs(s_est - s_true[0:len(s_est)])/s_true[0:len(s_est)]

<jax.Array float32(10,) ≈0.12 ±0.3 [≥2.3e-06, ≤1.0] nonzero:10
  <Arrayviz rendering>
| Device: GPU 0>

In [290]:
s_est

<jax.Array float32(10,) ≈2.4 ±3.7 [≥8.7e-05, ≤1.2e+01] nonzero:10
  <Arrayviz rendering>
| Device: GPU 0>

# Write simple algorthmic function and profile 

In [291]:
from functools import partial

In [302]:
@partial(jax.jit, static_argnames=['r'])
def rand_svd(A, r, key):
    m, n = A.shape
    
    keys = jax.random.split(key, r)
    
    Omega = jnp.stack([jax.random.normal(keys[i], (n,)) for i in range(r + 1)], axis=1)

    Q = jnp.zeros((m, 0))
    for j in range(1, r+1):    
        y_j = jnp.dot(A, Omega[:, j])        
        y_j = y_j - jnp.dot(Q, jnp.dot(Q.T, y_j))
        y_j = y_j / jnp.linalg.norm(y_j)
        Q = jnp.hstack((Q, y_j[:, None]))

    # With Q calculated, get the svd 
    # Step 4: Form the k x n matrix B = Q^T A
    B = Q.T @ A
    
    # Step 5
    Uhat, s_est, vh_est = jnp.linalg.svd(B)
    
    # Step 6
    U_est = Q @ Uhat

    return s_est

@partial(jax.jit, static_argnames=['r'])
def rand_svd_matrix_free(jvp_func, vjp_func, r, key):    
    keys = jax.random.split(key, r)
    
    # Need structure and 
    temp, back = jax.flatten_util.ravel_pytree(params)
    n = len(temp)

    # Generate 
    Omega = jnp.stack([jax.random.normal(keys[i], (n,)) for i in range(r + 1)], axis=1)
    
    Q = jnp.zeros((m, 0))
    for j in range(1, r+1):    
        _, y_j = jax.jvp(lambda p: model.apply(p, x), (params,), (back(Omega[:, j]),)) 
        y_j = jnp.squeeze(y_j)
        # print(y_j, j)
        # y_j = jnp.dot(A, Omega[:, j])      
        # print(y_j, j)

        y_j = y_j - jnp.dot(Q, jnp.dot(Q.T, y_j))
        y_j = y_j / jnp.linalg.norm(y_j)
        Q = jnp.hstack((Q, y_j[:, None]))

    # With Q calculated, get the svd 
    # Step 4: Form the k x n matrix B = Q^T A
    # B = Q.T @ A
    def apply_vjp_and_flatten(column): # Define a function that applies vjp_fn to a single column and flattens the result
        column = column[:, jnp.newaxis]  # Reshape to (16, 1)
        flat, _ = jax.flatten_util.ravel_pytree(vjp_fn(column)[0])
        return flat
    
    # Use vmap to vectorize the function over the columns of Q
    B = jax.vmap(apply_vjp_and_flatten, in_axes=1, out_axes=0)(Q)    
    
    # Step 5
    Uhat, s_est, vh_est = jnp.linalg.svd(B)
    
    # Step 6
    U_est = Q @ Uhat

    return U_est, s_est, vh_est

In [293]:
key = jax.random.PRNGKey(0)

print(rand_svd(jacobian, 5, key))

key = jax.random.PRNGKey(0)
_, jvp_output = jax.jvp(lambda p: model.apply(p, x), (params,), (params,)) 
_, vjp_fn = jax.vjp(f, params, x)
print(rand_svd_matrix_free(jvp_output, vjp_fn, 5, key))

[1.1682007e+01 6.7225695e+00 1.9332584e+00 1.1811699e+00 6.0362029e-03]
[1.1684262e+01 6.7298584e+00 1.9435952e+00 1.1738859e+00 1.4038066e-03]


In [294]:
%timeit rand_svd_matrix_free(jvp_output, vjp_fn, 5, key).block_until_ready()

578 μs ± 6.26 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [295]:
%timeit rand_svd(jacobian, 5, key).block_until_ready()

428 μs ± 3.56 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [303]:
u, s, vh = rand_svd_matrix_free(jvp_output, vjp_fn, 5, key)

In [306]:
vh.shape

(361, 361)

In [300]:
(rand_svd_matrix_free(jvp_output, vjp_fn, 10, key)[0:5] - s_true[0:5]) / s_true[0:5]


<jax.Array float32(5,) ≈0.00048 ±0.00064 [≥-3.8e-05, ≤0.0016] nonzero:5
  <Arrayviz rendering>
| Device: GPU 0>

In [301]:
s_est

<jax.Array float32(10,) ≈2.4 ±3.7 [≥8.7e-05, ≤1.2e+01] nonzero:10
  <Arrayviz rendering>
| Device: GPU 0>

In [87]:
%timeit rand_svd(jacobian, 32, key).block_until_ready()

1.33 ms ± 14.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [88]:
%timeit rand_svd(jacobian.T, 32, key).block_until_ready()

1.46 ms ± 63.5 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [89]:
rand_svd(jacobian.T, 32, key)

<jax.Array float32(15,) ≈1.6 ±3.2 [≥0.00023, ≤1.2e+01] nonzero:15
  <Arrayviz rendering>
| Device: GPU 0>

In [91]:
s_est = rand_svd(jacobian.T, 32, key)
(s_true[0:len(s_est)] - s_est) / s_true[0:len(s_est)]

<jax.Array float32(15,) ≈-0.047 ±0.34 [≥-0.99, ≤0.67] nonzero:15
  <Arrayviz rendering>
| Device: GPU 0>

In [93]:
s_est

<jax.Array float32(15,) ≈1.6 ±3.2 [≥0.00023, ≤1.2e+01] nonzero:15
  <Arrayviz rendering>
| Device: GPU 0>

In [94]:
s_true

<jax.Array float32(16,) ≈1.5 ±3.1 [≥3.3e-05, ≤1.2e+01] nonzero:16
  <Arrayviz rendering>
| Device: GPU 0>